# Options Pricing

In [45]:
import numpy as np
import pandas as pd
import scipy.stats as sp
import matplotlib.pyplot as plt
from stockdex import Ticker

## Stock Details

In [78]:
def volatility(stock):
    """Calculates the volatility of a stock, pulling 1 month of data from Yahoo Finance, with a 1d granularity"""
    if type(stock) != str:
        print("Please enter a valid stock code")
        return None
    ticker = Ticker(ticker=stock)
    price = ticker.yahoo_api_price(range='1mo', dataGranularity='1d')
    price_arr = price['close']
    pct_chgs = price_arr.pct_change().dropna()  # Calculate % changes 
    std = np.std(pct_chgs)
    volatility = std*np.sqrt(252) #Annualized volatility from 1mo of data.
    return volatility

In [80]:
#AAPL
S0 = 200 #initial price
K = 120 #strike price
T = 1 #Time to maturity in years
r = 0.04258 #risk free rate: us treasury 10 year bond yield 22/3/25
sigma = volatility("AAPL") #volatility

## Black Scholes Functions

In [81]:
def black_scholes_call(S0,K,T,r,sigma):
    """Calculates call price using Black-Scholes Options pricing model"""
    d1 = (np.log(S0/K)+(r+((sigma**2)/2))*T)/(sigma*np.sqrt(T))
    d2 = d1 - (sigma*np.sqrt(T))
    price = S0*sp.norm.cdf(d1) - K*np.exp(-r*T)*sp.norm.cdf(d2)
    return price

In [82]:
def black_scholes_put(S0,K,T,r,sigma):
    """Calculates put price using Black-Scholes Options pricing model"""
    d1 = (np.log(S0/K)+(r+((sigma**2)/2))*T)/(sigma*np.sqrt(T))
    d2 = d1 - (sigma*np.sqrt(T))
    price = K*np.exp(-r*T)*sp.norm.cdf(-d2) - S0*sp.norm.cdf(-d1)
    return price

In [83]:
black_scholes_put(S0,K,T,r,sigma)

0.5412263588782933

## Binomial Tree Functions

In [21]:
def binomial_tree_call(S0,K,T,r,sigma,N=10000):
    """Docstring"""
    dt = T/N
    u = np.exp(sigma*np.sqrt(dt))
    d = np.exp(-sigma*np.sqrt(dt))
    p_u = (np.exp(r*dt)-d)/(u-d)
    p_d = 1-p_u
    
    #empty array for price tree
    prices = np.zeros((N+1, N+1))
    for i in range(N+1): #Step number
        for j in range(i+1): #Node within that step
            prices[j,i] = S0 * (u**j) * (d**(i-j)) #j and i-j so they count in opposite directions (one up to i+1, one down from i to 0)
                                                    #All down, to all up.
    #Finds all possible payoffs
    option_values = np.maximum(prices[:, N]-K,0) #if price - K is less than 0, sets payoff to zero. (wouldn't use option)
    
    #loop backwards from final price to find best price for now
    for k in range(N-1, -1, -1):
        for l in range(k+1): #possible nodes at step l
            #https://users.physics.ox.ac.uk/~Foot/Phynance/Binomial2013.pdf
            # exp term accounts for risk free return. Rest is standard expected value calculation.
            option_values[l] = np.exp(-r*dt)*((p_u*option_values[l])+((p_d)*option_values[l+1]))
            
    return option_values[0] #returns value at node 0 (the start, ie. how you should price it)

In [22]:
def binomial_tree_put(S0,K,T,r,sigma,N=10000):
    """Docstring"""
    dt = T/N
    u = np.exp(sigma*np.sqrt(dt))
    d = np.exp(-sigma*np.sqrt(dt))
    p_u = (np.exp(r*dt)-d)/(u-d)
    p_d = 1-p_u
    
    #empty array for price tree
    prices = np.zeros((N+1, N+1))
    for i in range(N+1): #Step number
        for j in range(i+1): #Node within that step
            prices[j,i] = S0 * (u**j) * (d**(i-j)) #j and i-j so they count in opposite directions (one up to i+1, one down from i to 0)
                                                    #All down, to all up.
    #Finds all possible payoffs
    option_values = np.maximum(K - prices[:, N],0) #if K - price is less than 0, sets payoff to zero. (wouldn't use option)
    
    #loop backwards from final price to find best price for now
    for k in range(N-1, -1, -1):
        for l in range(k+1): #possible nodes at step l
            #https://users.physics.ox.ac.uk/~Foot/Phynance/Binomial2013.pdf
            # exp term accounts for risk free return. Rest is standard expected value calculation.
            option_values[l] = np.exp(-r*dt)*((p_u*option_values[l])+((p_d)*option_values[l+1]))
            
    return option_values[0] #returns value at node 0 (the start, ie. how you should price it)

In [23]:
binomial_tree_put(S0,K,T,r,sigma,N=10000)

0.39731575174077316

In [ ]:
## Monte Carlo Functions